In [1]:
import numpy as np
import pandas as pd

np.random.seed(42)

In [2]:
# cap at 50000 for POC or else takes too long
df = pd.read_csv('used_cars_data.csv', nrows=50000)
df = df[df['is_new'] == False].copy()

In [3]:
features = ['year', 'mileage', 'make_name', 'body_type', 'fuel_type', 'transmission', 'wheel_system']
target = 'price'

df = df[features + [target]].copy()

# drop rows missing price or mileage
df.dropna(subset=[target, 'mileage'], inplace=True)

# fill missing categoricals + one hot encode
cat_cols = ['make_name', 'body_type', 'fuel_type', 'transmission', 'wheel_system']
df[cat_cols] = df[cat_cols].fillna('Unknown')
df_encoded = pd.get_dummies(df, columns=cat_cols, drop_first=True)

X_raw = df_encoded.drop(columns=[target]).values.astype(float)
y = df_encoded[target].values.astype(float)

In [4]:
# standardize X and y
X_mean = X_raw.mean(axis=0)
X_std = X_raw.std(axis=0) + 1e-8
X = (X_raw - X_mean) / X_std

y_mean = y.mean()
y_std = y.std()
y_scaled = (y - y_mean) / y_std

# 80/20 train/test split
n = X.shape[0]
split = int(0.8 * n)
idx = np.random.permutation(n)

X_train = X[idx[:split]]
X_test  = X[idx[split:]]
y_train = y_scaled[idx[:split]]
y_test  = y_scaled[idx[split:]]

In [5]:
d = X_train.shape[1]
q = 64
eta = 0.01

# the rows of W1 correspond to the columns of X, the columns to the number of hidden nodes
# the rows of W2 correspond to the number of hidden nodes, the columns to the dimension of the output y
W1 = np.random.randn(d, q) * 0.01
W2 = np.random.randn(q, 1) * 0.01

In [6]:
# uses the ReLU activation function for h and the linear activation function for the output
def f(x):
    h = np.maximum(0, W1.T.dot(x))
    return W2.T.dot(h)

In [7]:
# keep track of gradient descent errors
errors = []
epochs = 200
n_train = X_train.shape[0]

for epoch in range(epochs):
    dW2 = 0
    for i, j in enumerate(y_train):
        x = np.reshape(X_train[i], (d, 1))
        h = np.maximum(0, W1.T.dot(x))
        dW2 += (2/n_train) * (f(x) - y_train[i]) * h

    W2 = W2 - eta * dW2

    dW1 = 0
    for i, j in enumerate(y_train):
        x = np.reshape(X_train[i], (d, 1))
        h = np.maximum(0, W1.T.dot(x))
        mat1 = np.heaviside(h, 0)
        dW1 += (2/n_train) * (f(x) - y_train[i]) * np.kron(x, (W2 * mat1).T)

    W1 = W1 - eta * dW1
    e = (1/n_train) * np.sum(np.square(f(X_train.T) - y_train))
    errors.append(e)

In [8]:
y_pred_scaled = f(X_test.T).flatten()
y_pred = y_pred_scaled * y_std + y_mean
y_actual = y_test * y_std + y_mean

mae = np.mean(np.abs(y_pred - y_actual))
ss_res = np.sum((y_actual - y_pred)**2)
ss_tot = np.sum((y_actual - y_actual.mean())**2)
r2 = 1 - ss_res / ss_tot

print(mae)
print(r2)

9725.073830444751
0.2557819545643162
